# Topic 3: Schema Management

> **Phase 5 → Week 8 → Topic 3**

---

## Why Schema Management Matters

In production ETL, data sources change. A vendor adds a column. A system sends a new field. An upstream team renames a key. Without proper schema management, your pipeline breaks or silently corrupts data.

---

## Interview Cheat Sheet

**Q: How do you define a schema in Spark?**
> Two ways: (1) `StructType` in Python — explicit and type-safe: `StructType([StructField('col', StringType(), nullable=True), ...])`. (2) DDL string — compact: `'col1 STRING, col2 INT, col3 DOUBLE'`. Both produce the same result. Use `StructType` for programmatic schemas (built in code), DDL strings for quick definitions.

**Q: What is schema enforcement vs schema evolution?**
> Schema **enforcement** = reject or coerce data that doesn't match the defined schema. Schema **evolution** = accept new data that has a slightly different schema (new columns added, columns reordered) and merge it with the existing schema. Delta Lake supports both: it enforces by default and allows evolution with `mergeSchema=True` or `ALTER TABLE ADD COLUMN`.

**Q: What happens when you read a CSV column as `IntegerType` but the data contains strings?**
> In `PERMISSIVE` mode (default), the value becomes `null` and the row is included in `_corrupt_record` if you set that column. In `FAILFAST` mode, it throws immediately. In `DROPMALFORMED` mode, the row is dropped. This is why explicit schemas + `PERMISSIVE` + capturing `_corrupt_record` is the production pattern.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week8 - Schema Management") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: Defining Schemas

In [ ]:
# Method 1: StructType (most expressive — use for production)
order_schema = StructType([
    StructField("order_id",    StringType(),    nullable=False),
    StructField("customer_id", StringType(),    nullable=True),
    StructField("amount",      DecimalType(10, 2), nullable=True),  # precise money
    StructField("status",      StringType(),    nullable=True),
    StructField("order_date",  DateType(),      nullable=True),
    StructField("tags",        ArrayType(StringType()), nullable=True),
    StructField("metadata",    MapType(StringType(), StringType()), nullable=True),
])

print("StructType schema:")
print(order_schema.simpleString())

# Method 2: DDL string (compact, good for simple schemas)
ddl_schema = "order_id STRING NOT NULL, customer_id STRING, amount DECIMAL(10,2), status STRING, order_date DATE"
schema_from_ddl = spark.createDataFrame([], ddl_schema).schema
print("\nDDL schema:")
print(schema_from_ddl.simpleString())

# Method 3: StructType from JSON (useful for storing schemas in config)
schema_json = order_schema.json()
restored_schema = StructType.fromJson(__import__('json').loads(schema_json))
print("\nSchema round-tripped through JSON (same as original):")
print(restored_schema == order_schema)

## Part 2: Type Coercion and Casting

In [ ]:
# Cast at read time vs transform time
raw = spark.createDataFrame([
    ("O001", "250.00",  "2024-01-15", "1"),
    ("O002", "89.50",   "2024-01-16", "0"),
    ("O003", "INVALID", "bad-date",   "1"),
    ("O004", "45.00",   "2024-01-18", None),
], ["order_id", "amount_str", "date_str", "is_vip_str"])

# Safe casting — invalid values become null (not exception)
typed = raw \
    .withColumn("amount",   F.col("amount_str").cast(DoubleType())) \
    .withColumn("order_date", F.to_date("date_str", "yyyy-MM-dd")) \
    .withColumn("is_vip",   F.col("is_vip_str").cast(BooleanType()))

print("After casting (invalid → null, not exception):")
typed.select("order_id", "amount", "order_date", "is_vip").show()

# Detect cast failures
cast_failures = typed.filter(
    F.col("amount").isNull() & F.col("amount_str").isNotNull()
)
print(f"Cast failures (non-null input → null output): {cast_failures.count()} rows")
cast_failures.select("order_id", "amount_str", "amount").show()

## Part 3: Schema Validation Patterns

In [ ]:
# Pattern: validate incoming schema against expected schema

def validate_schema(df, expected_schema, strict=False):
    """
    Validate a DataFrame schema against an expected StructType.
    strict=True: exact match (no extra columns allowed)
    strict=False: all expected columns must exist with correct types
    Returns (is_valid, list_of_errors)
    """
    errors = []
    actual_fields = {f.name: f for f in df.schema.fields}
    expected_fields = {f.name: f for f in expected_schema.fields}

    # Check all expected columns exist
    for name, field in expected_fields.items():
        if name not in actual_fields:
            errors.append(f"MISSING column: {name} ({field.dataType})")
        elif actual_fields[name].dataType != field.dataType:
            errors.append(f"TYPE MISMATCH: {name} expected {field.dataType} got {actual_fields[name].dataType}")

    # In strict mode, no extra columns allowed
    if strict:
        for name in actual_fields:
            if name not in expected_fields:
                errors.append(f"UNEXPECTED column: {name}")

    return len(errors) == 0, errors


# Test the validator
expected = StructType([
    StructField("order_id",  StringType(),  True),
    StructField("amount",    DoubleType(),  True),
    StructField("status",    StringType(),  True),
])

correct_df = spark.createDataFrame([], "order_id STRING, amount DOUBLE, status STRING")
wrong_df   = spark.createDataFrame([], "order_id STRING, amount STRING, region STRING")

ok, errs = validate_schema(correct_df, expected)
print(f"Correct schema valid: {ok}")

ok, errs = validate_schema(wrong_df, expected)
print(f"Wrong schema valid: {ok}")
for e in errs:
    print(f"  ✗ {e}")

In [ ]:
# Pattern: add missing columns with default null values (schema padding)
def align_schema(df, target_schema):
    """Add missing columns (as null) and reorder to match target schema."""
    existing = {f.name for f in df.schema.fields}
    for field in target_schema.fields:
        if field.name not in existing:
            df = df.withColumn(field.name, F.lit(None).cast(field.dataType))
    # Select in target order
    return df.select([f.name for f in target_schema.fields
                      if f.name in {f2.name for f2 in df.schema.fields}])

# Simulate upstream sending incomplete schema
incoming = spark.createDataFrame([
    ("O001", 250.0),
    ("O002", 89.5),
], ["order_id", "amount"])

full_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("amount",   DoubleType(), True),
    StructField("status",   StringType(), True),
    StructField("region",   StringType(), True),
])

aligned = align_schema(incoming, full_schema)
print("After schema alignment (missing columns added as null):")
aligned.show()

## Part 4: Nested Schemas

In [ ]:
# Complex nested schema definition
nested_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer", StructType([
        StructField("id",   StringType(), True),
        StructField("name", StringType(), True),
        StructField("address", StructType([
            StructField("city",    StringType(), True),
            StructField("country", StringType(), True),
        ]), True),
    ]), True),
    StructField("items", ArrayType(StructType([
        StructField("sku",   StringType(), True),
        StructField("qty",   IntegerType(), True),
        StructField("price", DoubleType(), True),
    ])), True),
    StructField("attributes", MapType(StringType(), StringType()), True),
])

# Create a DataFrame with this nested schema
from pyspark.sql import Row

data = [
    Row(
        order_id="O001",
        customer=Row(id="C1", name="Alice", address=Row(city="NYC", country="USA")),
        items=[Row(sku="SKU1", qty=2, price=49.99), Row(sku="SKU2", qty=1, price=19.99)],
        attributes={"priority": "high", "channel": "web"}
    )
]

nested_df = spark.createDataFrame(data, nested_schema)
nested_df.printSchema()

# Access nested fields
nested_df.select(
    "order_id",
    "customer.name",
    "customer.address.city",
    F.col("items")[0]["sku"].alias("first_sku"),
    F.col("attributes")["channel"].alias("channel")
).show()

## Part 5: Flattening Nested Schemas

In [ ]:
# Utility: auto-flatten one level of nested StructType fields
def flatten_struct(df, col_name=None):
    """Flatten all StructType columns in a DataFrame (one level deep)."""
    new_cols = []
    for field in df.schema.fields:
        if isinstance(field.dataType, StructType):
            for subfield in field.dataType.fields:
                new_cols.append(
                    F.col(f"`{field.name}`.`{subfield.name}`")
                     .alias(f"{field.name}_{subfield.name}")
                )
        else:
            new_cols.append(F.col(field.name))
    return df.select(new_cols)

# Flatten nested customer struct
flat_df = flatten_struct(nested_df)
print("After flatten_struct (one level):")
flat_df.printSchema()

# Flatten again to get nested.nested
flat_df2 = flatten_struct(flat_df)
print("After second flatten (two levels deep):")
flat_df2.show(truncate=False)

## Exercises

1. Write a `StructType` schema for the orders CSV at `/workspace/week4/data/orders.csv`. Include proper types for each column (not all `StringType`).
2. Write a `validate_schema()` function that raises a `ValueError` if any expected column is missing or has the wrong type.
3. You receive a JSON file where the `amount` field is sometimes a string (`"99.99"`) and sometimes a number (`99.99`). How do you handle this safely in Spark?
4. Explain the difference between `nullable=True` and `nullable=False` in a `StructField`. Does Spark enforce `nullable=False` at runtime?
5. Create a nested schema for a `user_events` table: each event has `event_id`, `user_id`, `timestamp`, and `properties` (a map of string to string). Write and read it back.

In [ ]:
# Exercise 3: Mixed string/number in JSON
import json as json_lib

mixed_json = [
    {"order_id": "O1", "amount": 99.99},
    {"order_id": "O2", "amount": "149.50"},  # string!
    {"order_id": "O3", "amount": 75},
]
with open("/tmp/mixed_amounts.json", "w") as f:
    for r in mixed_json:
        f.write(json_lib.dumps(r) + "\n")

# Strategy: read as string, cast after
safe_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("amount",   StringType(), True),  # read as string first
])

df_mixed = spark.read.schema(safe_schema).json("/tmp/mixed_amounts.json")
df_typed = df_mixed.withColumn("amount_num", F.col("amount").cast(DoubleType()))

print("Mixed string/number amounts — safe handling:")
df_typed.show()
print("All values successfully cast to Double regardless of source type")

# Answer Q4: nullable enforcement
print("""
Answer Q4: nullable=False
  Spark does NOT enforce nullable=False at runtime.
  It is a HINT to the optimizer, not a runtime constraint.
  If data actually contains nulls in a 'NOT NULL' column,
  Spark will process them without error.
  To truly enforce non-null: add an explicit filter or assertion:
    df.filter(col('order_id').isNotNull())  # real enforcement
""")